In [6]:
import os 
from minsearch import Index
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv() # get api key 

openai_client = OpenAI(
    api_key=os.getenv("CEREBRAS_API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase


In [21]:
messages = [
    {"role": "system", "content": "I just discovered the course. Can I join it?"}
]
response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages =messages,
)
output_text = response.choices[0].message.content
print(output_text)

I would be happy to help! However, I don't know which specific course you are referring to.

To give you the right answer, could you please tell me:

1.  **The name of the course?**
2.  **A link to the course page?**
3.  **Where you found it** (e.g., Coursera, Udemy, a university website, etc.)?

Once you provide that information, I can check if it is currently open for enrollment and how you can sign up.


In [ ]:
documents = load_faq_data()
index = build_index(documents)


query = "how do i run olama locally ? "

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )


search(query)

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': 'fe8fed31e6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'How do I get token counts for Module 1 homework if I use a different provider?',
  'answer': "For the current Module 1 homework, get the token

In [31]:
search_tool = {
    "type": "function",
    "function": {  # <--- This nested key is what was missing!
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [ ]:
response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages =messages,
    tools = [search_tool] #llm uses to carry on task from the user
)
print(response)

ChatCompletion(id='chatcmpl-19ef7aab-7ef4-471a-a507-c6ef14491bdb', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='1edb05209', function=Function(arguments='{"query": "join course enroll registration"}', name='search'), type='function')], reasoning='The user is asking about joining a course they just discovered. This sounds like they want to know about enrollment or registration options. I should search the FAQ for information about how to join or enroll in a course.'))], created=1782133734, model='zai-glm-4.7', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_a33c042689e969567008', usage=CompletionUsage(completion_tokens=56, prompt_tokens=196, total_tokens=252, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=N